# Generate Indian Ride-Sharing Dataset
This notebook generates 50,000 synthetic ride-sharing records representing Indian market patterns across major metro cities, ride tiers, and operators (Ola & Uber India).

In [ ]:
import numpy as np
import pandas as pd
import os

# Set seed for reproducibility
np.random.seed(42)
num_records = 50000
print(f"Generating {num_records} records...")

## Feature Definitions & Distributions
- **Cities (city_encoded):** Mumbai (0), Delhi (1), Bangalore (2), Hyderabad (3), Chennai (4), Pune (5)
- **Cab Operator (cab_type_encoded):** Ola (0), Uber India (1)
- **Ride Tiers (name_encoded):** Auto (0), Mini (1), Sedan (2), Prime Sedan (3), Prime SUV (4), Bike (5)
- **Distances:** Distance brackets tailored per category (e.g. short Auto/Bike trips vs. long Prime SUV/Airport trips)

In [ ]:
# 1. Ride Tiers
name_encoded = np.random.choice([0, 1, 2, 3, 4, 5], size=num_records, p=[0.25, 0.25, 0.20, 0.15, 0.10, 0.05])

# 2. Cab Operator
cab_type_encoded = np.random.choice([0, 1], size=num_records, p=[0.55, 0.45])

# 3. Cities
city_encoded = np.random.choice([0, 1, 2, 3, 4, 5], size=num_records, p=[0.25, 0.20, 0.20, 0.15, 0.10, 0.10])

# 4. Airport routes (10% of overall rides)
is_airport = np.random.choice([False, True], size=num_records, p=[0.90, 0.10])

# 5. Distances
distance_km = np.zeros(num_records)
for i in range(num_records):
    if is_airport[i]:
        distance_km[i] = np.random.uniform(20.0, 45.0)
    else:
        tier = name_encoded[i]
        if tier in [0, 5]: # Auto, Bike
            distance_km[i] = np.random.uniform(1.0, 8.0)
        elif tier in [1, 2]: # Mini, Sedan
            distance_km[i] = np.random.uniform(2.0, 25.0)
        else: # Prime Sedan, Prime SUV
            distance_km[i] = np.random.uniform(5.0, 60.0)
distance_km = np.round(distance_km, 2)

# 6. Time Fields
pickup_hour = np.random.randint(0, 24, size=num_records)
pickup_day = np.random.randint(0, 7, size=num_records) # 0=Mon, 6=Sun
pickup_month = np.random.randint(1, 13, size=num_records)
pickup_day_of_month = np.random.randint(1, 29, size=num_records)

# 7. Weather
is_bad_weather = np.random.choice([False, True], size=num_records, p=[0.85, 0.15])

## Price Generation (Base Fare & Rates)
Based on standard Indian market rates:
- Auto (₹8/km, ₹15 base)
- Mini (₹10/km, ₹30 base)
- Sedan (₹12/km, ₹35 base)
- Prime Sedan (₹15/km, ₹45 base)
- Prime SUV (₹20/km, ₹60 base)
- Bike (₹6/km, ₹10 base)

In [ ]:
base_rates = {0: 8.0, 1: 10.0, 2: 12.0, 3: 15.0, 4: 20.0, 5: 6.0}
base_fares = {0: 15.0, 1: 30.0, 2: 35.0, 3: 45.0, 4: 60.0, 5: 10.0}

price = np.zeros(num_records)
for i in range(num_records):
    tier = name_encoded[i]
    price[i] = base_fares[tier] + distance_km[i] * base_rates[tier]
price = np.round(price, 2)

## Surge Multiplier Logic
Encoding real Indian surge rules:
1. Peak hours (8-11am, 6-10pm weekdays) → surge 1.3x to 2.5x
2. Monsoon months (June to September) → surge +0.3x
3. Festival dates (Diwali, New Year, Holi) → surge 2.0x to 3.5x
4. Late night (11pm-5am) → surge 1.4x to 1.8x
5. Sunday evening (hour 17-21) → surge 1.2x to 1.6x
6. Metro cities (Mumbai, Delhi, Bangalore) → base surge +0.1x
7. Airport routes (>25km) → surge +0.2x
8. Rain simulation (is_bad_weather=True) → surge +0.2x

In [ ]:
is_festival = np.zeros(num_records, dtype=bool)
surge_multiplier = np.ones(num_records)

for i in range(num_records):
    h = pickup_hour[i]
    d = pickup_day[i]
    m = pickup_month[i]
    dom = pickup_day_of_month[i]
    city = city_encoded[i]
    dist = distance_km[i]
    bad_w = is_bad_weather[i]
    
    # Check festival status
    is_fest = False
    if m == 1 and dom == 1: # New Year
        is_fest = True
    elif m == 3 and dom in [1, 2, 3]: # Holi
        is_fest = True
    elif m in [10, 11] and dom in [1, 2, 3, 4, 5]: # Diwali
        is_fest = True
    is_festival[i] = is_fest
    
    # Baseline surge multiplier
    s = 1.0
    
    if is_fest:
        s = np.random.uniform(2.0, 3.5)
    else:
        # Time-based multipliers
        if d < 5 and ((8 <= h <= 11) or (18 <= h <= 21)): # Weekday Peak hours (8-11am, 6-10pm)
            s = np.random.uniform(1.3, 2.5)
        elif h >= 23 or h < 5: # Late night (11pm-5am)
            s = np.random.uniform(1.4, 1.8)
        elif d == 6 and (17 <= h <= 21): # Sunday evening (hour 17-21)
            s = np.random.uniform(1.2, 1.6)
            
    # Additive modifiers
    if m in [6, 7, 8, 9]: # Monsoon months
        s += 0.3
        
    if city in [0, 1, 2]: # Metro cities (Mumbai, Delhi, Bangalore)
        s += 0.1
        
    if dist > 25: # Airport / long route
        s += 0.2
        
    if bad_w: # Rain simulation
        s += 0.2
        
    # Clamping and noise
    s = np.clip(s, 1.0, 5.0)
    if not is_fest and s == 1.0:
        pass
    else:
        s += np.random.normal(0, 0.05)
        s = np.clip(s, 1.0, 5.0)
        
    surge_multiplier[i] = np.round(s, 2)

## Construct DataFrame and Save to CSV
Creating the pandas DataFrame and writing it to `indian_rides.csv`.

In [ ]:
df = pd.DataFrame({
    'distance_km': distance_km,
    'pickup_hour': pickup_hour,
    'pickup_day': pickup_day,
    'pickup_month': pickup_month,
    'price': price,
    'cab_type_encoded': cab_type_encoded,
    'name_encoded': name_encoded,
    'city_encoded': city_encoded,
    'is_bad_weather': is_bad_weather.astype(int),
    'is_festival': is_festival.astype(int),
    'surge_multiplier': surge_multiplier
})

# Save to destination path
output_path = 'indian_rides.csv'
df.to_csv(output_path, index=False)
print(f"Data saved successfully to {output_path}")

## Statistical Analytics & Validation
Shape, distribution statistics, and correlation matrix analysis.

In [ ]:
print("1. DataFrame Shape:")
print(df.shape)

print("\n2. Surge Multiplier Distribution Statistics:")
print(df['surge_multiplier'].describe())

print("\n3. Correlation Matrix with Target Variable (surge_multiplier):")
corr = df.corr()
print(corr['surge_multiplier'].sort_values(ascending=False))